# 03 · Invisible failures (W3) — the no-reaction limit

WildChat-100K under the coupling lens: failure visibility, depth correction, archetypes, triggers, repair.
**Backs:** `docs/safety/w2w3-recoding.md`, W3 section of `docs/shared-findings.md`, `docs/reference/invisible-failures-summary.md`.
**Scripts:** `predecessor_stats.compute()`, `w3_coupling_stats.compute()`.

In [1]:
import sys, pathlib, json
ROOT = pathlib.Path.cwd(); ROOT = ROOT if (ROOT/"src").exists() else ROOT.parent
sys.path.insert(0, str(ROOT/"src"))
import pandas as pd
pd.set_option("display.max_rows", 120)
print("repo root:", ROOT)

repo root: /data/wang/junh/githubs/human-agent-coupling-errors


## Headline — failure visibility
Doc claim: **62.6%** of 100K conversations contain a failure; **78.9%** of failures draw NO user reaction (invisible). Counts: none **37,443** / invisible **49,368** / visible **7,632** / mixed **5,557**.

In [2]:
import predecessor_stats as ps
res = ps.compute()   # reads the 34 MB predecessor annotations once
h = res["headline"]
print(f"N={h['N']:,} | failures={h['fails']:,} ({h['failure_rate_pct']:.1f}%) | invisible/fail={h['invisible_pct_of_fails']:.1f}%")
pd.DataFrame([h["visibility"]])

N=100,000 | failures=62,557 (62.6%) | invisible/fail=78.9%


,none,invisible,visible,mixed
0,37443,49368,7632,5557


In [3]:
pd.DataFrame(h["quality"]).T  # quality grades

,n,pct
good,36888.0,36.888
acceptable,41002.0,41.002
poor,18177.0,18.177
critical,3933.0,3.933


## Depth correction — the single-turn artifact
Doc claim: multi-turn (≥2) invisible/fail = **49.5%**; ≥4 turns drop further. Single-turn mass inflates the 78.9% headline.

In [4]:
pd.DataFrame(res["depth"]["rows"])

,label,n,failure_rate_pct,invisible_pct_of_fails
0,turns == 1,63398,58.116975,99.430045
1,turns == 2,14746,60.619829,66.025282
2,turns == 3,7506,69.517719,51.897279
3,turns 4-9,12092,79.110155,37.894627
4,turns >= 10,2258,88.086802,25.037707
5,multi-turn (>=2),36602,70.247527,49.521624


## Archetypes
Doc claim: `the_walkaway` **94.3%** of failures, `the_silent_mismatch` **55.1%**, `the_confidence_trap` **35.0%**; mean 1.67 archetypes/labeled conv.

In [5]:
pd.DataFrame(res["archetypes"]["prevalence"])

,archetype,n,pct_of_fails,pct_of_all
0,the_walkaway,58995,94.305993,58.995
1,none,36598,58.503445,36.598
2,the_silent_mismatch,34454,55.076171,34.454
3,the_confidence_trap,21868,34.956919,21.868
4,the_partial_recovery,8386,13.405374,8.386
5,the_death_spiral,3464,5.537350,3.464
6,the_drift,2916,4.661349,2.916
7,the_contradiction_unravel,301,0.481161,0.301
8,the_mystery_failure,8,0.012788,0.008


In [6]:
pd.DataFrame(res["archetypes"]["domain_archetype"])  # domain x archetype PPMI

,domain,archetype,ppmi
0,translation_language,the_mystery_failure,2.393490
1,personal_lifestyle,the_mystery_failure,2.019279
2,software_development,the_death_spiral,0.832446
3,translation_language,the_contradiction_unravel,0.829514
4,general_knowledge,the_contradiction_unravel,0.811070
5,it_infrastructure,the_contradiction_unravel,0.799591
6,education_academic,the_contradiction_unravel,0.683454
7,personal_lifestyle,the_drift,0.652651
8,it_infrastructure,the_partial_recovery,0.620818
9,translation_language,the_confidence_trap,0.598425


## Walkaway deep-dive
Doc claim: `the_walkaway` present in **98.0%** of invisible failures; ~90% co-occur with a substantive cause (walkaway is a *visibility marker*, not the cause).

In [7]:
res["walkaway"]

{'n_invisible': 49368,
 'invisible_with_walkaway': 48393,
 'invisible_with_walkaway_pct': 98.02503646086534,
 'invisible_walkaway_only': 4589,
 'invisible_walkaway_only_pct': 9.295495057527143,
 'mix_excl_walkaway': [{'archetype': 'the_silent_mismatch',
   'n': 34454,
   'pct_of_fails': 55.076170532474386},
  {'archetype': 'the_confidence_trap',
   'n': 21868,
   'pct_of_fails': 34.95691928960788},
  {'archetype': 'the_partial_recovery',
   'n': 8386,
   'pct_of_fails': 13.405374298639641},
  {'archetype': 'the_death_spiral',
   'n': 3464,
   'pct_of_fails': 5.537349936857586},
  {'archetype': 'the_drift', 'n': 2916, 'pct_of_fails': 4.661348849848937},
  {'archetype': 'the_contradiction_unravel',
   'n': 301,
   'pct_of_fails': 0.4811611810029253},
  {'archetype': 'the_mystery_failure',
   'n': 8,
   'pct_of_fails': 0.0127883370366226}],
 'failures_only_walkaway_or_none': 5062,
 'failures_only_walkaway_or_none_pct': 8.091820259922951}

## Signal prevalence (10K calibrated summaries)
Doc claim: `generate_without_clarifying` **44.1%**, `false_confidence` **26.6%**, `silent_assumption` **20.2%**.

In [8]:
pd.DataFrame(res["signals"]["ai_top"])

,signal,pct
0,conversation_advanced,86.00
1,intent_addressed,84.79
2,scope_matched,73.25
3,ai_structured_response,58.95
4,generate_without_clarifying,44.06
5,false_confidence,26.55
6,problem_ignored,26.18
7,under_delivered,26.09
8,factual_error,22.76
9,intent_missed,22.37


## Triggers → failure, and repair sparsity (W3)
Doc claim: base failure rate **57.4%**; `user_provides_invalid_input` lift **1.40×**; **67.1%** of failures have NO repair act by either party.

In [9]:
import w3_coupling_stats as w3
w = w3.compute()
tr = w["triggers_repair"]
print(f"base failure rate {tr['base_rate_pct']:.1f}% | NO-repair by either party {tr['no_repair_pct']:.1f}% "
      f"(AI {tr['ai_repair_present_pct']:.1f}%, human {tr['human_repair_present_pct']:.1f}%)")
pd.DataFrame(tr["triggers"])

base failure rate 57.4% | NO-repair by either party 67.1% (AI 20.7%, human 24.6%)


,trigger,prev_pct,p_fail_pct,lift
0,user_ambiguous_request,19.33,74.236937,1.294228
1,user_multi_request,21.22,75.824694,1.321909
2,user_scope_change,6.14,71.172638,1.240806
3,user_provides_invalid_input,4.46,80.044843,1.395482
